# Relevance scores

**Input:**  
- Gexf file (from Social Network Visualisation Tool)
- Channel files, raw from data provider (e.g. pandas dataframes or csv/xlsx/json files)

**Dependencies:** Social Network Visualisation Tool

In [17]:
%pip install pyiceberg==0.8.0 pyiceberg[s3fs,sql-sqlite]
!pip install torch torchvision torchaudio
!pip install transformers

import pandas as pd
import numpy as np
from collections import Counter
import networkx as nx
import transformers

Note: you may need to restart the kernel to use updated packages.


## Read graph

In [18]:
G = nx.read_gexf("graph.gexf")

## Read and filter data

In [19]:
df = pd.read_csv('network_df.csv')
df = df.dropna(axis=1, how='all')
df.shape

(452, 16)

In [20]:
# Recover channel language code (from SQL)
info = pd.read_csv('get_channels_query_info.csv')
info.rename(columns={
    'id': 'channel_id',
    'name': 'title'
},
    inplace = True)

# Recover parent ID (from  telegram.channels_to_query)
query_df = pd.read_csv('query_df.csv')
query_df.rename(columns={
    'id': 'channel_id'
},
    inplace = True)
query_df['channel_id'] = pd.to_numeric(query_df['channel_id'], errors='coerce').astype('Int64')

In [21]:
df['channel_id'] = df['channel_id'].astype(float).astype('Int64')
info['channel_id'] = info['channel_id'].astype(float).astype('Int64')

df = df.merge(info, how='inner', on=['channel_id', 'username', 'title'])
# df = df.merge(query_df[['username', 'parent_channel_id']], how='inner', on='username')
df.shape

(452, 19)

## Relevance score 1: language

In [22]:
project_languages = ["EN", "FR", "ES", "DE", "EL", "IT", "PL", "RO"]
df = df[df.language_code.isin(project_languages)]
df.shape

(452, 19)

## Relevance score 2: channel virality

Channel relevance is some metric for the combination of:  
* channel activity (a = number of posts);
* channel participation (p = number of participants)
* channel centrality (d = channel degree (total))

In [23]:
# -------------------------------
# Compute node degrees from graph
# -------------------------------
# Extract total_degree_w from node attributes
channel_total_degree_w = {n: float(attr.get('total_degree_w', 0)) for n, attr in G.nodes(data=True)}
# Assume G is a NetworkX graph with channel nodes labeled by channels['id']
df['channel_id'] = df['channel_id'].astype(float).astype(str)
df["total_degree_w"] = df["channel_id"].map(channel_total_degree_w).fillna(0)

# -------------------------------
# Compute channel virality score
# -------------------------------
# Cube root of product of participants_count, message_count, and node_degree
df["channel_virality_score"] = np.cbrt(
    df["participants_count"].fillna(0) *
    df["message_count"].fillna(0) *
    df["total_degree_w"].fillna(0)
)

# -------------------------------
# Mark top 10% channels as viral
# -------------------------------
threshold = np.percentile(df["channel_virality_score"], 90)
df["is_channel_viral"] = df["channel_virality_score"] >= threshold

# -------------------------------
# ✅ Inspect
# -------------------------------
df[["channel_id", "participants_count", "message_count", "total_degree_w", "channel_virality_score", "is_channel_viral"]].sample(5)
# print(f'Number of viral channels: {len(df[df.is_channel_viral == True])}')

,channel_id,participants_count,message_count,total_degree_w,channel_virality_score,is_channel_viral
223,1118285951.0,3519.0,999542.0,93.0,6890.241135,True
221,1118285951.0,3519.0,999542.0,93.0,6890.241135,True
311,1452083659.0,369.0,505327.0,52.0,2132.393767,False
3,1702605423.0,287.0,27003.0,2.0,249.330057,False
366,1783546642.0,1903.0,33985.0,8.0,802.796277,False


## Relevance score 3: post virality

In [24]:
# 1️⃣ Compute total reactions per message (already done)
df['reaction_sums'] = df['reactions'].apply(
    lambda x: sum(v for v in x.values() if v is not None) if isinstance(x, dict) else 0
)

# 2️⃣ Compute a virality score (simple sum of interactions)
df['message_virality_score'] = df['nr_replies'].fillna(0) + \
                        df['forwards'].fillna(0) + \
                        df['reaction_sums'].fillna(0)

# 3️⃣ Compute 90th percentile threshold
threshold = np.percentile(df['message_virality_score'], 90)

# 4️⃣ Mark viral messages
df['is_message_viral'] = df['message_virality_score'] >= threshold


## End-result

In [26]:
columns_to_keep = ['date', 'forwards', 'nr_replies', 'id', 'views', 'message', 'reactions', 'channel_id', 'fwd_from_id', 'about',
                   'participants_count', 'message_count', 'title', 'verified', 'username', 'query_id', 'total_degree_w',
                   'channel_virality_score', 'is_channel_viral', 'reaction_sums',
                   'message_virality_score' ,'is_message_viral']

df = df[columns_to_keep]

df = df.rename(columns={
    'id': 'message_id'
})

In [27]:
df.columns

Index(['date', 'forwards', 'nr_replies', 'message_id', 'views', 'message',
       'reactions', 'channel_id', 'fwd_from_id', 'about', 'participants_count',
       'message_count', 'title', 'verified', 'username', 'query_id',
       'total_degree_w', 'channel_virality_score', 'is_channel_viral',
       'reaction_sums', 'message_virality_score', 'is_message_viral'],
      dtype='object')

In [28]:
df.shape

(452, 22)

In [29]:
df.drop_duplicates().shape

(452, 22)

In [30]:
df.to_csv('Telegram_Relevance_Evaluator_Output.csv', index = False)

**Output:**  
- ```msg_to_keep``` (list of relevant messages in the raw text format)
- ```channels_to_keep``` (list of relevant channel ids)